# CANguard -- PIRD v2 Extensions (thin orchestrator)

Multi-scale windows, threshold sweeps, and bus-level / per-byte ablations,
all driven by the `canguard` library. See `pird_v2_extensions.ipynb` in git
history for the original exploratory analysis.


## 1. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from canguard.data import get_loader
from canguard.detectors import IsolationForestDetector
from canguard.evaluation import (
    train_anomaly_detector,
)
from canguard.features import (
    BEHAVIORAL_FEATURES_V1 as FEATURES,
)
from canguard.features import (
    FeaturePipeline,
    fit_known_ids_on_normal_prefix,
    fit_per_id_stats,
    temporal_split,
    transform_residuals,
)
from canguard.visualization import plot_threshold_sweep

plt.rcParams['figure.dpi'] = 120
DATA_DIR = Path('HCRL Car-Hacking')
SAMPLE_SIZE = 60000
NAMES = ['DoS', 'Fuzzy', 'RPM', 'gear']


## 2. Multi-scale windows (10 / 30 / 50)


In [ ]:
samples = {
    name: get_loader('hcrl', DATA_DIR / f'{name}_dataset.csv').load(sample_size=SAMPLE_SIZE)
    for name in NAMES
}
WINDOW_SIZES = [10, 30, 50]
tables = {}
for ws in WINDOW_SIZES:
    tables[ws] = {}
    for name, df in samples.items():
        known = fit_known_ids_on_normal_prefix(df)
        pipe = FeaturePipeline(window_size=ws, known_ids=known)
        tables[ws][name] = pipe.process_dataframe(df)
        pipe.reset()
        print(f'ws={ws:>2d} {name:>5s}: {len(tables[ws][name])} windows')


## 3. Residual IF per scale + threshold sweep


In [ ]:
scale_results = {}
sweep_tables = {}
for ws in WINDOW_SIZES:
    scale_results[ws] = {}
    for name in NAMES:
        ft = tables[ws][name]
        calib, train, test = temporal_split(ft, 0.4, 0.2, 0.4)
        stats, gstats = fit_per_id_stats(calib, FEATURES)
        res_cols = [c + '_res' for c in FEATURES]
        res_train = transform_residuals(train, stats, gstats, FEATURES)
        res_test = transform_residuals(test, stats, gstats, FEATURES)
        det = IsolationForestDetector(n_estimators=200, random_state=0)
        out = train_anomaly_detector(det, res_train, res_test, res_cols)
        scale_results[ws][name] = out
        print(f'ws={ws:>2d} {name:>5s}: F1={out["f1"]:.3f}  Recall={out["recall"]:.3f}')


## 4. Threshold sweep at ws=30 (best config)


In [ ]:
# For the sweep we refit once on train normals and reuse stored scores
from canguard.evaluation.threshold import sweep_thresholds

ws = 30
sweep_tables = {}
for name in ['RPM', 'DoS']:
    ft = tables[ws][name]
    calib, train, test = temporal_split(ft, 0.4, 0.2, 0.4)
    stats, gstats = fit_per_id_stats(calib, FEATURES)
    res_cols = [c + '_res' for c in FEATURES]
    res_train = transform_residuals(train, stats, gstats, FEATURES)
    res_test = transform_residuals(test, stats, gstats, FEATURES)
    train_norm = res_train[res_train['is_attack'] == 0].copy()
    n_val = max(1, int(len(train_norm) * 0.2))
    det = IsolationForestDetector(n_estimators=200, random_state=0)
    det.fit(train_norm.iloc[:-n_val][res_cols].fillna(0).values)
    val_scores = det.score_samples(train_norm.iloc[-n_val:][res_cols].fillna(0).values)
    test_scores = det.score_samples(res_test[res_cols].fillna(0).values)
    y_test = res_test['is_attack'].values
    rows = sweep_thresholds(val_scores, test_scores, y_test, [0.001, 0.01, 0.05])
    sweep_tables[name] = rows
    print(f'\n{name} sweep:'); print(pd.DataFrame(rows)[['target_FPR','actual_FPR','recall','f1']].to_string(index=False))


## 5. Threshold sweep plots


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, ['RPM', 'DoS']):
    plot_threshold_sweep(sweep_tables[name], title=name, ax=ax)
plt.tight_layout(); plt.show()


## 6. Supervised reference (HGB, supervised upper bound)


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

for name in ['RPM', 'DoS']:
    ft = tables[30][name]
    calib, train, test = temporal_split(ft, 0.4, 0.2, 0.4)
    stats, gstats = fit_per_id_stats(calib, FEATURES)
    res_cols = [c + '_res' for c in FEATURES]
    res_train = transform_residuals(train, stats, gstats, FEATURES)
    res_test = transform_residuals(test, stats, gstats, FEATURES)
    Xtr = res_train[res_cols].fillna(0).values
    ytr = res_train['is_attack'].values
    Xte = res_test[res_cols].fillna(0).values
    yte = res_test['is_attack'].values
    hgb = HistGradientBoostingClassifier(max_iter=200, random_state=0)
    hgb.fit(Xtr, ytr)
    print(f'\n=== {name} -- HGB (supervised) ===')
    print(classification_report(yte, hgb.predict(Xte), target_names=['Normal','Attack'], zero_division=0))


## Notes


Per-byte residual features and adaptive EMA baselines (v2 exploratory additions) are
implemented in the original `pird_v2_extensions.ipynb`. This thin notebook exposes the
core multi-scale + threshold-sweep results through the library API.
